# Demo SoundStream implementation

Download repo

In [1]:
!git clone https://github.com/112Brothers/SoundStream.git
!cd SoundStream

Cloning into 'SoundStream'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 79 (delta 25), reused 78 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 18.15 KiB | 364.00 KiB/s, done.
Resolving deltas: 100% (25/25), done.


Install requirements

In [2]:
!pip install torchmetrics[audio]

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.6 MB/s eta 0:00:00
  Created wheel for pesq: filename=pesq-0.0.4-cp312-cp312-linux_x86_64.whl size=284122 sha256=59975c16f2fdfdc0f4c9745aa03e93ebeddba8473d1c6cef582dcc72319483f4
  Stored in directory: /root/.cache/pip/wheels/9b/d4/a4/9cf3512534cd47ce4a036d1593ee4013f2bf7509e631a147a3
Successfully built pesq


Import

In [3]:
import sys, torch, numpy as np
import librosa, librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sys.path.insert(0, "SoundStream/src")

Load model

In [4]:
from huggingface_hub import hf_hub_download
from evaluate import load_model

ckpt_path = hf_hub_download(repo_id="alexok2006/SoundStream_Implementation", filename="ckpt_best_stoi.pt")
device = "cuda" if torch.cuda.is_available() else "cpu"
model, step = load_model(ckpt_path, device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ckpt_best_stoi.pt:   0%|          | 0.00/320M [00:00<?, ?B/s]

Model: C=32, D=500, K=512, num_q=8
Generator params: 8.08M


Download wav-file and convert it:

In [18]:
import librosa, torch, urllib.request, tempfile, os

SR = 16000
def load_audio(src):
  with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
      urllib.request.urlretrieve(src, tmp.name)
      wav_np, sr = librosa.load(tmp.name, mono=True)
      os.remove(tmp.name)
  if sr != SR:
      wav_np = librosa.resample(wav_np, orig_sr=sr, target_sr=SR)
      sr = SR
  wav = torch.from_numpy(wav_np).float().unsqueeze(0).unsqueeze(0)
  return wav

def inference(audio):
  with torch.no_grad():
    x = audio.to(device)
    rec, *_ = model(x)
    rec = rec.squeeze(0).cpu().numpy()
  return rec

Show sample

In [22]:

sample = load_audio("https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav") # url...
rec = inference(sample)
print("Orig:")
display(Audio(sample.squeeze().cpu().numpy(), rate=SR)) # orig
print("Rec:")
display(Audio(rec, rate=SR))

Orig:


Rec:
